# Dyck Language Experiment

Tests whether transformers learn stack-state tracking (balanced parentheses) from next-token prediction.

**Runtime:** Select GPU via Runtime > Change runtime type > T4 GPU

**Estimated time:** ~15-20 min on T4 (D1 + D2, 3 seeds each, 10k train)

In [ ]:
# Cell 1: Check GPU + Mount Google Drive
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f'Memory: {total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

# Results will be saved here
DRIVE_OUTPUT = '/content/drive/MyDrive/fibonacci-experiment/experiments_v2/exp_dyck'
import os
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'\nResults will save to Google Drive: {DRIVE_OUTPUT}')

In [ ]:
%%writefile model.py
"""Small GPT-style transformer for learning Fibonacci sequences."""
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import List, Dict, Optional


class NumberTokenizer:
    def __init__(self, vocabulary: List[int]):
        self.vocabulary = sorted(vocabulary)
        self.pad_token = -1
        self.unk_token = -2
        self.token_to_id = {num: idx for idx, num in enumerate(self.vocabulary)}
        self.token_to_id[self.pad_token] = len(self.vocabulary)
        self.token_to_id[self.unk_token] = len(self.vocabulary) + 1
        self.id_to_token = {idx: num for num, idx in self.token_to_id.items()}
        self.vocab_size = len(self.token_to_id)
        self.pad_id = self.token_to_id[self.pad_token]
        self.unk_id = self.token_to_id[self.unk_token]

    def encode(self, numbers: List[int]) -> List[int]:
        return [self.token_to_id.get(num, self.unk_id) for num in numbers]

    def decode(self, token_ids: List[int]) -> List[int]:
        return [self.id_to_token.get(tid, self.unk_token) for tid in token_ids]


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 100):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1), :]


class FibonacciTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=3,
                 dim_feedforward=512, dropout=0.1, max_seq_len=50):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.dropout(x)
        if src_mask is None:
            seq_len = src.size(1)
            src_mask = self._generate_square_subsequent_mask(seq_len).to(src.device)
        x = self.transformer(x, src_mask, src_key_padding_mask=src_key_padding_mask,
                             is_causal=True)
        logits = self.fc_out(x)
        return logits

    def _generate_square_subsequent_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
%%writefile experiment_framework.py
"""Minimal experiment framework — just seeding."""
import random
import numpy as np
import torch

DATA_SEED = 0
RANDOM_SEEDS = [42, 123, 7]

def set_all_seeds(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
%%writefile exp5_baselines.py
"""Baselines — N-gram and kNN (minimal for Dyck experiment)."""
from collections import defaultdict
from typing import List, Tuple


class NgramBaseline:
    """N-gram next-token model with backoff and Laplace smoothing."""
    def __init__(self, n: int, smoothing: float = 1.0):
        self.n = n
        self.smoothing = smoothing
        self.counts = {}
        self.vocab = set()

    def train(self, examples: List[Tuple[List[int], int]]):
        for ctx, tgt in examples:
            self.vocab.add(tgt)
            self.vocab.update(ctx)
            for order in range(self.n, 0, -1):
                if order not in self.counts:
                    self.counts[order] = defaultdict(lambda: defaultdict(int))
                if len(ctx) >= order:
                    key = tuple(ctx[-order:])
                    self.counts[order][key][tgt] += 1
        if 0 not in self.counts:
            self.counts[0] = defaultdict(lambda: defaultdict(int))
        for ctx, tgt in examples:
            self.counts[0][()][tgt] += 1

    def predict(self, context: List[int]) -> int:
        for order in range(self.n, -1, -1):
            if order not in self.counts:
                continue
            if len(context) >= order:
                key = tuple(context[-order:]) if order > 0 else ()
                if key in self.counts[order] and self.counts[order][key]:
                    token_counts = self.counts[order][key]
                    V = len(self.vocab)
                    best_token, best_score = None, -1
                    for token in self.vocab:
                        score = (token_counts.get(token, 0) + self.smoothing) / (
                            sum(token_counts.values()) + self.smoothing * V)
                        if score > best_score:
                            best_score = score
                            best_token = token
                    if best_token is not None:
                        return best_token
        if self.vocab:
            return max(self.vocab, key=lambda t: self.counts.get(0, {}).get((), {}).get(t, 0))
        return 0


class KNNBaseline:
    """kNN retrieval baseline using Hamming distance."""
    def __init__(self, k: int = 5, context_window: int = 10):
        self.k = k
        self.context_window = context_window
        self.contexts = []
        self.targets = []

    def train(self, examples: List[Tuple[List[int], int]]):
        for ctx, tgt in examples:
            if len(ctx) > self.context_window:
                ctx = ctx[-self.context_window:]
            elif len(ctx) < self.context_window:
                ctx = [0] * (self.context_window - len(ctx)) + ctx
            self.contexts.append(ctx)
            self.targets.append(tgt)

    def predict(self, context: List[int]) -> int:
        if len(context) > self.context_window:
            context = context[-self.context_window:]
        elif len(context) < self.context_window:
            context = [0] * (self.context_window - len(context)) + context
        dists = [(sum(1 for x, y in zip(context, c) if x != y), t)
                 for c, t in zip(self.contexts, self.targets)]
        dists.sort(key=lambda x: x[0])
        vote_counts = defaultdict(int)
        for _, t in dists[:self.k]:
            vote_counts[t] += 1
        return max(vote_counts, key=vote_counts.get)

In [ ]:
%%writefile exp_dyck.py
"""
Dyck Language Experiment — Balanced Parentheses as Formal Language Control

Tests whether transformers can learn stack-state tracking (a fundamentally
compositional operation) from next-token prediction on Dyck-1 and Dyck-2.
"""
import argparse
import json
import os
import random
import sys
import time
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from model import FibonacciTransformer, count_parameters
from experiment_framework import DATA_SEED, RANDOM_SEEDS, set_all_seeds
from exp5_baselines import NgramBaseline, KNNBaseline

OUTPUT_DIR = 'experiments_v2/exp_dyck'


def _print(msg: str = ''):
    """Print with immediate flush."""
    print(msg, flush=True)


# ===========================================================================
# Tokenizer
# ===========================================================================
class DyckTokenizer:
    """Tokenizer for Dyck-1 and Dyck-2 languages."""
    def __init__(self, bracket_types: int = 1):
        assert bracket_types in (1, 2)
        self.bracket_types = bracket_types
        self.PAD = 0
        self.BOS = 1
        self.EOS = 2
        self.OPEN_PAREN = 3
        self.CLOSE_PAREN = 4
        self.id_to_str = {0: 'PAD', 1: 'BOS', 2: 'EOS', 3: '(', 4: ')'}
        self.str_to_id = {'PAD': 0, 'BOS': 1, 'EOS': 2, '(': 3, ')': 4}
        if bracket_types == 2:
            self.OPEN_BRACKET = 5
            self.CLOSE_BRACKET = 6
            self.id_to_str[5] = '['
            self.id_to_str[6] = ']'
            self.str_to_id['['] = 5
            self.str_to_id[']'] = 6
            self.vocab_size = 7
        else:
            self.vocab_size = 5
        self.pad_id = self.PAD

    def open_ids(self) -> List[int]:
        ids = [self.OPEN_PAREN]
        if self.bracket_types == 2:
            ids.append(self.OPEN_BRACKET)
        return ids

    def close_ids(self) -> List[int]:
        ids = [self.CLOSE_PAREN]
        if self.bracket_types == 2:
            ids.append(self.CLOSE_BRACKET)
        return ids

    def matching_close(self, open_id: int) -> int:
        if open_id == self.OPEN_PAREN:
            return self.CLOSE_PAREN
        if self.bracket_types == 2 and open_id == self.OPEN_BRACKET:
            return self.CLOSE_BRACKET
        raise ValueError(f'Not an open bracket ID: {open_id}')

    def matching_open(self, close_id: int) -> int:
        if close_id == self.CLOSE_PAREN:
            return self.OPEN_PAREN
        if self.bracket_types == 2 and close_id == self.CLOSE_BRACKET:
            return self.OPEN_BRACKET
        raise ValueError(f'Not a close bracket ID: {close_id}')

    def is_open(self, token_id: int) -> bool:
        return token_id in self.open_ids()

    def is_close(self, token_id: int) -> bool:
        return token_id in self.close_ids()

    def encode(self, tokens: List[str]) -> List[int]:
        return [self.str_to_id[t] for t in tokens]

    def decode(self, ids: List[int]) -> List[str]:
        return [self.id_to_str.get(i, '?') for i in ids]

    def decode_string(self, ids: List[int]) -> str:
        return ''.join(self.decode(ids))


# ===========================================================================
# Data Generation
# ===========================================================================
def generate_dyck_string(tokenizer, n_pairs, max_depth, rng):
    tokens = [tokenizer.BOS]
    stack = []
    pairs_remaining = n_pairs
    while pairs_remaining > 0 or len(stack) > 0:
        depth = len(stack)
        if depth >= max_depth or pairs_remaining == 0:
            tokens.append(tokenizer.matching_close(stack.pop()))
        elif depth == 0:
            open_id = rng.choice(tokenizer.open_ids())
            tokens.append(open_id)
            stack.append(open_id)
            pairs_remaining -= 1
        else:
            p_open = pairs_remaining / (pairs_remaining + depth)
            if rng.random() < p_open:
                open_id = rng.choice(tokenizer.open_ids())
                tokens.append(open_id)
                stack.append(open_id)
                pairs_remaining -= 1
            else:
                tokens.append(tokenizer.matching_close(stack.pop()))
    tokens.append(tokenizer.EOS)
    return tokens


def generate_split(tokenizer, n_sequences, min_pairs, max_pairs, max_depth, seed):
    rng = random.Random(seed)
    sequences = []
    for _ in range(n_sequences):
        n_pairs = rng.randint(min_pairs, max_pairs)
        seq = generate_dyck_string(tokenizer, n_pairs, max_depth, rng)
        sequences.append(seq)
    return sequences


def compute_data_distributions(sequences, tokenizer):
    lengths = []
    max_depths = []
    open_counts_by_pos = defaultdict(int)
    close_counts_by_pos = defaultdict(int)
    total_by_pos = defaultdict(int)
    for seq in sequences:
        content = seq[1:-1]
        lengths.append(len(content))
        depth = 0
        max_d = 0
        for tok in content:
            if tokenizer.is_open(tok):
                depth += 1
                max_d = max(max_d, depth)
            elif tokenizer.is_close(tok):
                depth -= 1
        max_depths.append(max_d)
        for i, tok in enumerate(content):
            total_by_pos[i] += 1
            if tokenizer.is_open(tok):
                open_counts_by_pos[i] += 1
            elif tokenizer.is_close(tok):
                close_counts_by_pos[i] += 1
    length_hist = {}
    for l in lengths:
        length_hist[l] = length_hist.get(l, 0) + 1
    depth_hist = {}
    for d in max_depths:
        depth_hist[d] = depth_hist.get(d, 0) + 1
    open_ratio_by_pos = {}
    for pos in range(min(80, max(total_by_pos.keys()) + 1 if total_by_pos else 0)):
        if total_by_pos[pos] > 0:
            open_ratio_by_pos[pos] = open_counts_by_pos[pos] / total_by_pos[pos]
    return {
        'n_sequences': len(sequences),
        'mean_length': float(np.mean(lengths)) if lengths else 0,
        'std_length': float(np.std(lengths)) if lengths else 0,
        'min_length': int(min(lengths)) if lengths else 0,
        'max_length': int(max(lengths)) if lengths else 0,
        'mean_max_depth': float(np.mean(max_depths)) if max_depths else 0,
        'std_max_depth': float(np.std(max_depths)) if max_depths else 0,
        'length_histogram': {str(k): v for k, v in sorted(length_hist.items())},
        'depth_histogram': {str(k): v for k, v in sorted(depth_hist.items())},
        'open_ratio_by_position': {str(k): round(v, 3) for k, v in sorted(open_ratio_by_pos.items())},
    }


# ===========================================================================
# Validation
# ===========================================================================
def is_valid_dyck(token_ids, tokenizer):
    stack = []
    for i, tok in enumerate(token_ids):
        if tok == tokenizer.BOS:
            continue
        if tok == tokenizer.EOS:
            if len(stack) > 0:
                return False, i, 'unclosed'
            return True, None, None
        if tok == tokenizer.PAD:
            continue
        if tokenizer.is_open(tok):
            stack.append(tok)
        elif tokenizer.is_close(tok):
            if len(stack) == 0:
                return False, i, 'underflow'
            expected_close = tokenizer.matching_close(stack[-1])
            if tok != expected_close:
                return False, i, 'wrong_type'
            stack.pop()
        else:
            return False, i, 'unexpected_token'
    if len(stack) > 0:
        return False, len(token_ids), 'unclosed'
    return True, None, None


# ===========================================================================
# Oracle
# ===========================================================================
def oracle_next_tokens(token_ids, tokenizer, max_pairs_in_sequence=40):
    legal_sets = []
    stack = []
    total_opens = 0
    for i, tok in enumerate(token_ids[:-1]):
        if tok == tokenizer.BOS:
            pass
        elif tokenizer.is_open(tok):
            stack.append(tok)
            total_opens += 1
        elif tokenizer.is_close(tok):
            if stack:
                stack.pop()
        depth = len(stack)
        legal = set()
        if depth < 20 and total_opens < max_pairs_in_sequence:
            legal.update(tokenizer.open_ids())
        if depth > 0:
            legal.add(tokenizer.matching_close(stack[-1]))
        if depth == 0 and total_opens > 0:
            legal.add(tokenizer.EOS)
        legal_sets.append(legal)
    return legal_sets


def compute_oracle_stats(sequences, tokenizer):
    total_positions = 0
    forced_positions = 0
    for seq in sequences:
        legal_sets = oracle_next_tokens(seq, tokenizer, max_pairs_in_sequence=40)
        for legal_set in legal_sets:
            total_positions += 1
            if len(legal_set) == 1:
                forced_positions += 1
    return {
        'total_positions': total_positions,
        'forced_positions': forced_positions,
        'ambiguous_positions': total_positions - forced_positions,
        'forced_fraction': forced_positions / total_positions if total_positions > 0 else 0,
    }


# ===========================================================================
# LM Dataset
# ===========================================================================
class DyckLMDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long)


def collate_right_pad(batch, pad_id=0):
    max_len = max(t.size(0) for t in batch)
    inputs, targets, masks = [], [], []
    for seq in batch:
        seq_len = seq.size(0)
        inp = seq[:-1]
        tgt = seq[1:]
        pad_len = max_len - seq_len
        if pad_len > 0:
            inp = torch.cat([inp, torch.full((pad_len,), pad_id, dtype=torch.long)])
            tgt = torch.cat([tgt, torch.full((pad_len,), pad_id, dtype=torch.long)])
        mask = (inp == pad_id)
        inputs.append(inp)
        targets.append(tgt)
        masks.append(mask)
    return torch.stack(inputs), torch.stack(targets), torch.stack(masks)


# ===========================================================================
# Training
# ===========================================================================
def train_lm(train_sequences, tokenizer, random_seed, output_dir,
             epochs=50, batch_size=32, lr=0.001, d_model=128, nhead=4,
             num_layers=3, dim_feedforward=512, dropout=0.1, max_seq_len=256,
             device=None, verbose=True):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    set_all_seeds(random_seed)
    os.makedirs(output_dir, exist_ok=True)
    dataset = DyckLMDataset(train_sequences)
    pad_id = tokenizer.PAD
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        collate_fn=lambda b: collate_right_pad(b, pad_id))
    model = FibonacciTransformer(
        vocab_size=tokenizer.vocab_size, d_model=d_model, nhead=nhead,
        num_layers=num_layers, dim_feedforward=dim_feedforward,
        dropout=dropout, max_seq_len=max_seq_len).to(device)
    if verbose:
        _print(f'    Model parameters: {count_parameters(model):,}')
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    best_loss = float('inf')
    history = {'losses': [], 'accuracies': []}
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        for inputs, targets, padding_mask in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            padding_mask = padding_mask.to(device)
            optimizer.zero_grad()
            logits = model(inputs, src_key_padding_mask=padding_mask)
            logits_flat = logits.reshape(-1, tokenizer.vocab_size)
            targets_flat = targets.reshape(-1)
            loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=pad_id)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            non_pad = targets_flat != pad_id
            if non_pad.any():
                preds = logits_flat.argmax(dim=-1)
                correct += (preds[non_pad] == targets_flat[non_pad]).sum().item()
                total += non_pad.sum().item()
        avg_loss = total_loss / len(loader)
        accuracy = 100.0 * correct / total if total > 0 else 0.0
        history['losses'].append(avg_loss)
        history['accuracies'].append(accuracy)
        scheduler.step(avg_loss)
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'loss': avg_loss, 'accuracy': accuracy},
                       os.path.join(output_dir, 'best_model.pt'))
        if verbose and (epoch + 1) % 10 == 0:
            _print(f'      Epoch {epoch+1}/{epochs}: loss={avg_loss:.4f}, acc={accuracy:.2f}%')
    if verbose:
        _print(f'      Training complete. Best loss={best_loss:.4f}, Final acc={accuracy:.2f}%')
    ckpt = torch.load(os.path.join(output_dir, 'best_model.pt'), map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, history


# ===========================================================================
# Evaluation: Teacher-Forced
# ===========================================================================
def evaluate_teacher_forced_dyck(model, sequences, tokenizer, device=None):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval()
    total_correct = 0
    total_tokens = 0
    exact_matches = 0
    first_errors = []
    depth_correct = defaultdict(int)
    depth_total = defaultdict(int)
    open_correct = 0
    open_total = 0
    close_correct = 0
    close_total = 0
    illegal_predictions = 0
    underflow_predictions = 0
    wrong_type_predictions = 0
    with torch.no_grad():
        for seq in sequences:
            seq_tensor = torch.tensor([seq], dtype=torch.long, device=device)
            inp = seq_tensor[:, :-1]
            tgt = seq_tensor[:, 1:]
            logits = model(inp)
            preds = logits.argmax(dim=-1).squeeze(0)
            targets = tgt.squeeze(0)
            stack = []
            seq_correct = True
            first_err = None
            for pos in range(targets.size(0)):
                target_tok = targets[pos].item()
                pred_tok = preds[pos].item()
                if target_tok == tokenizer.PAD:
                    continue
                depth = len(stack)
                total_tokens += 1
                depth_total[depth] += 1
                if pred_tok == target_tok:
                    total_correct += 1
                    depth_correct[depth] += 1
                else:
                    if seq_correct:
                        first_err = pos
                        seq_correct = False
                if tokenizer.is_open(target_tok):
                    open_total += 1
                    if pred_tok == target_tok:
                        open_correct += 1
                elif tokenizer.is_close(target_tok):
                    close_total += 1
                    if pred_tok == target_tok:
                        close_correct += 1
                if tokenizer.is_close(pred_tok):
                    if depth == 0:
                        illegal_predictions += 1
                        underflow_predictions += 1
                    elif tokenizer.matching_close(stack[-1]) != pred_tok:
                        illegal_predictions += 1
                        wrong_type_predictions += 1
                if tokenizer.is_open(target_tok):
                    stack.append(target_tok)
                elif tokenizer.is_close(target_tok):
                    if stack:
                        stack.pop()
            if seq_correct:
                exact_matches += 1
            if first_err is not None:
                first_errors.append(first_err)
    n_sequences = len(sequences)
    return {
        'per_token_accuracy': 100.0 * total_correct / total_tokens if total_tokens > 0 else 0,
        'exact_match_rate': 100.0 * exact_matches / n_sequences if n_sequences > 0 else 0,
        'mean_first_error': float(np.mean(first_errors)) if first_errors else None,
        'per_depth_accuracy': {
            d: 100.0 * depth_correct[d] / depth_total[d]
            for d in sorted(depth_total.keys())
        },
        'open_accuracy': 100.0 * open_correct / open_total if open_total > 0 else 0,
        'close_accuracy': 100.0 * close_correct / close_total if close_total > 0 else 0,
        'illegal_prediction_rate': 100.0 * illegal_predictions / total_tokens if total_tokens > 0 else 0,
        'underflow_rate': 100.0 * underflow_predictions / total_tokens if total_tokens > 0 else 0,
        'wrong_type_rate': 100.0 * wrong_type_predictions / total_tokens if total_tokens > 0 else 0,
        'total_tokens': total_tokens,
        'n_sequences': n_sequences,
    }


# ===========================================================================
# Evaluation: Free-Run
# ===========================================================================
def evaluate_free_run(model, tokenizer, n_samples=200, max_len=100,
                      device=None, temperature=0.0):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval()
    valid_count = 0
    lengths = []
    error_types = defaultdict(int)
    first_errors = []
    samples = []
    with torch.no_grad():
        for i in range(n_samples):
            generated = [tokenizer.BOS]
            for step in range(max_len):
                inp = torch.tensor([generated], dtype=torch.long, device=device)
                logits = model(inp)
                next_logits = logits[0, -1, :]
                if temperature == 0:
                    next_tok = next_logits.argmax().item()
                else:
                    probs = F.softmax(next_logits / temperature, dim=-1)
                    next_tok = torch.multinomial(probs, 1).item()
                generated.append(next_tok)
                if next_tok == tokenizer.EOS:
                    break
            valid, err_pos, err_type = is_valid_dyck(generated, tokenizer)
            if valid:
                valid_count += 1
            else:
                if err_type:
                    error_types[err_type] += 1
                if err_pos is not None:
                    first_errors.append(err_pos)
            content_len = len(generated) - 2
            lengths.append(content_len)
            if i < 20:
                samples.append({
                    'ids': generated,
                    'string': tokenizer.decode_string(generated),
                    'valid': valid,
                    'error_type': err_type,
                    'error_pos': err_pos,
                })
    return {
        'validity_rate': 100.0 * valid_count / n_samples if n_samples > 0 else 0,
        'mean_length': float(np.mean(lengths)) if lengths else 0,
        'std_length': float(np.std(lengths)) if lengths else 0,
        'error_type_counts': dict(error_types),
        'mean_first_error': float(np.mean(first_errors)) if first_errors else None,
        'n_samples': n_samples,
        'temperature': temperature,
        'samples': samples,
    }


# ===========================================================================
# Baselines
# ===========================================================================
class BigramBaseline:
    def __init__(self):
        self.counts = defaultdict(lambda: defaultdict(int))
        self.vocab = set()
    def train(self, examples):
        for ctx, tgt in examples:
            last_tok = ctx[-1] if ctx else 0
            self.counts[last_tok][tgt] += 1
            self.vocab.add(tgt)
    def predict(self, context):
        last_tok = context[-1] if context else 0
        if last_tok in self.counts and self.counts[last_tok]:
            return max(self.counts[last_tok], key=self.counts[last_tok].get)
        if self.vocab:
            return next(iter(self.vocab))
        return 0


def sequences_to_ngram_examples(sequences, context_window=10):
    examples = []
    for seq in sequences:
        for i in range(1, len(seq)):
            ctx_start = max(0, i - context_window)
            ctx = seq[ctx_start:i]
            tgt = seq[i]
            examples.append((ctx, tgt))
    return examples


def evaluate_baseline_dyck(baseline, sequences, tokenizer, context_window=10):
    total_correct = 0
    total_tokens = 0
    exact_matches = 0
    for seq in sequences:
        seq_correct = True
        for i in range(1, len(seq)):
            ctx_start = max(0, i - context_window)
            ctx = seq[ctx_start:i]
            target = seq[i]
            if target == tokenizer.PAD:
                continue
            pred = baseline.predict(ctx)
            total_tokens += 1
            if pred == target:
                total_correct += 1
            else:
                seq_correct = False
        if seq_correct:
            exact_matches += 1
    return {
        'per_token_accuracy': 100.0 * total_correct / total_tokens if total_tokens > 0 else 0,
        'exact_match_rate': 100.0 * exact_matches / len(sequences) if sequences else 0,
        'total_tokens': total_tokens,
    }


# ===========================================================================
# Orchestration
# ===========================================================================
def run_dyck_experiment(bracket_types=1, n_train=10000, n_test=2000,
                        random_seeds=None, pilot=False, device=None):
    if random_seeds is None:
        random_seeds = RANDOM_SEEDS
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    label = f'D{bracket_types}'
    _print(f'\n{"="*70}')
    _print(f'  Dyck-{bracket_types} Experiment')
    _print(f'{"="*70}')
    tokenizer = DyckTokenizer(bracket_types=bracket_types)
    _print(f'  Vocab size: {tokenizer.vocab_size}')
    _print(f'  Tokens: {tokenizer.id_to_str}')
    if pilot:
        n_train = 2000
        n_test = 500
    _print(f'\n  Generating data (train={n_train}, test={n_test} per split)...')
    train_seqs = generate_split(tokenizer, n_train, 1, 20, 6, seed=DATA_SEED)
    id_test_seqs = generate_split(tokenizer, n_test, 1, 20, 6, seed=DATA_SEED + 1)
    longer_ood_seqs = generate_split(tokenizer, n_test, 30, 40, 14, seed=DATA_SEED + 2)
    deeper_ood_seqs = generate_split(tokenizer, n_test, 1, 20, 14, seed=DATA_SEED + 3)
    combined_ood_seqs = generate_split(tokenizer, n_test, 30, 40, 14, seed=DATA_SEED + 4)
    splits = {
        'train': train_seqs,
        'id_test': id_test_seqs,
        'longer_ood': longer_ood_seqs,
        'deeper_ood': deeper_ood_seqs,
        'combined_ood': combined_ood_seqs,
    }
    _print('  Validating all generated sequences...')
    for split_name, seqs in splits.items():
        invalid = sum(1 for seq in seqs if not is_valid_dyck(seq, tokenizer)[0])
        assert invalid == 0, f'FATAL: {invalid} invalid sequences in {split_name}'
        _print(f'    {split_name}: {len(seqs)} sequences, all valid')
    _print('  Computing data distributions...')
    distributions = {}
    for split_name, seqs in splits.items():
        distributions[split_name] = compute_data_distributions(seqs, tokenizer)
        d = distributions[split_name]
        _print(f'    {split_name}: mean_len={d["mean_length"]:.1f}, '
               f'mean_depth={d["mean_max_depth"]:.1f}')
    _print('  Computing oracle statistics...')
    oracle_stats = {}
    for split_name, seqs in splits.items():
        oracle_stats[split_name] = compute_oracle_stats(seqs, tokenizer)
        o = oracle_stats[split_name]
        _print(f'    {split_name}: forced={o["forced_fraction"]:.3f} '
               f'({o["forced_positions"]}/{o["total_positions"]})')
    # --- Baselines ---
    _print('\n  Training baselines...')
    ctx_win = 10
    train_examples = sequences_to_ngram_examples(train_seqs, context_window=ctx_win)
    baseline_results = {}
    # Bigram
    bigram = BigramBaseline()
    bigram.train(train_examples)
    for split_name, seqs in splits.items():
        if split_name == 'train':
            continue
        res = evaluate_baseline_dyck(bigram, seqs, tokenizer, context_window=ctx_win)
        baseline_results.setdefault('bigram', {})[split_name] = res
    _print(f'    Bigram ID: {baseline_results["bigram"]["id_test"]["per_token_accuracy"]:.2f}%')
    # N-gram
    for n in [3, 5]:
        ngram = NgramBaseline(n=n)
        ngram.train(train_examples)
        for split_name, seqs in splits.items():
            if split_name == 'train':
                continue
            res = evaluate_baseline_dyck(ngram, seqs, tokenizer, context_window=ctx_win)
            baseline_results.setdefault(f'ngram_{n}', {})[split_name] = res
        _print(f'    N-gram({n}) ID: {baseline_results[f"ngram_{n}"]["id_test"]["per_token_accuracy"]:.2f}%')
    # kNN — subsampled for speed
    knn_eval_max = 100
    knn_train_max = 5000
    for knn_ctx in [10, 20, 40]:
        knn_examples = sequences_to_ngram_examples(train_seqs, context_window=knn_ctx)
        if len(knn_examples) > knn_train_max:
            rng_knn = random.Random(DATA_SEED)
            knn_examples = rng_knn.sample(knn_examples, knn_train_max)
        knn = KNNBaseline(k=5, context_window=knn_ctx)
        knn.train(knn_examples)
        for split_name, seqs in splits.items():
            if split_name == 'train':
                continue
            eval_seqs = seqs[:knn_eval_max]
            res = evaluate_baseline_dyck(knn, eval_seqs, tokenizer, context_window=knn_ctx)
            baseline_results.setdefault(f'knn_ctx{knn_ctx}', {})[split_name] = res
            _print(f'    kNN(ctx={knn_ctx}) {split_name}: '
                   f'{res["per_token_accuracy"]:.2f}% (n={len(eval_seqs)})')
    # --- Transformer training ---
    _print(f'\n  Training transformers (seeds={random_seeds})...')
    transformer_results = {}
    for rs in random_seeds:
        run_dir = os.path.join(OUTPUT_DIR, f'{label}_RS{rs}')
        _print(f'\n    --- RS={rs} ---')
        model, history = train_lm(
            train_sequences=train_seqs, tokenizer=tokenizer, random_seed=rs,
            output_dir=run_dir, epochs=50, batch_size=32, lr=0.001,
            d_model=128, nhead=4, num_layers=3, dim_feedforward=512,
            dropout=0.1, max_seq_len=256, device=device, verbose=True)
        seed_results = {'history': history}
        for split_name, seqs in splits.items():
            if split_name == 'train':
                continue
            tf_res = evaluate_teacher_forced_dyck(model, seqs, tokenizer, device)
            seed_results[f'teacher_forced_{split_name}'] = tf_res
            _print(f'      TF {split_name}: token_acc={tf_res["per_token_accuracy"]:.2f}%, '
                   f'exact={tf_res["exact_match_rate"]:.2f}%, '
                   f'illegal={tf_res["illegal_prediction_rate"]:.2f}%')
        fr_greedy = evaluate_free_run(model, tokenizer, n_samples=200, max_len=100,
                                      device=device, temperature=0.0)
        seed_results['free_run_greedy'] = fr_greedy
        _print(f'      Free-run greedy: validity={fr_greedy["validity_rate"]:.2f}%, '
               f'mean_len={fr_greedy["mean_length"]:.1f}')
        fr_sampled = evaluate_free_run(model, tokenizer, n_samples=200, max_len=100,
                                       device=device, temperature=0.8)
        seed_results['free_run_sampled'] = fr_sampled
        _print(f'      Free-run sampled: validity={fr_sampled["validity_rate"]:.2f}%, '
               f'mean_len={fr_sampled["mean_length"]:.1f}')
        manifest = {
            'label': label, 'bracket_types': bracket_types,
            'random_seed': rs, 'vocab_size': tokenizer.vocab_size,
            'n_train': len(train_seqs), 'results': seed_results,
        }
        with open(os.path.join(run_dir, 'manifest.json'), 'w') as f:
            json.dump(manifest, f, indent=2, default=str)
        transformer_results[rs] = seed_results
    # --- Aggregate ---
    _print(f'\n  Aggregating transformer results across seeds...')
    agg_transformer = {}
    for split_name in ['id_test', 'longer_ood', 'deeper_ood', 'combined_ood']:
        tf_key = f'teacher_forced_{split_name}'
        accs = [transformer_results[rs][tf_key]['per_token_accuracy'] for rs in random_seeds]
        exact = [transformer_results[rs][tf_key]['exact_match_rate'] for rs in random_seeds]
        illegal = [transformer_results[rs][tf_key]['illegal_prediction_rate'] for rs in random_seeds]
        underflow = [transformer_results[rs][tf_key]['underflow_rate'] for rs in random_seeds]
        wrong_type = [transformer_results[rs][tf_key]['wrong_type_rate'] for rs in random_seeds]
        agg_transformer[split_name] = {
            'mean_token_acc': float(np.mean(accs)),
            'std_token_acc': float(np.std(accs)),
            'mean_exact_match': float(np.mean(exact)),
            'std_exact_match': float(np.std(exact)),
            'mean_illegal_rate': float(np.mean(illegal)),
            'std_illegal_rate': float(np.std(illegal)),
            'mean_underflow_rate': float(np.mean(underflow)),
            'std_underflow_rate': float(np.std(underflow)),
            'mean_wrong_type_rate': float(np.mean(wrong_type)),
            'std_wrong_type_rate': float(np.std(wrong_type)),
            'per_seed_token_acc': {str(rs): a for rs, a in zip(random_seeds, accs)},
        }
        _print(f'    {split_name}: token_acc={agg_transformer[split_name]["mean_token_acc"]:.2f} '
               f'\u00b1 {agg_transformer[split_name]["std_token_acc"]:.2f}%, '
               f'illegal={agg_transformer[split_name]["mean_illegal_rate"]:.2f}%')
    greedy_validities = [transformer_results[rs]['free_run_greedy']['validity_rate'] for rs in random_seeds]
    sampled_validities = [transformer_results[rs]['free_run_sampled']['validity_rate'] for rs in random_seeds]
    agg_transformer['free_run_greedy'] = {
        'mean_validity': float(np.mean(greedy_validities)),
        'std_validity': float(np.std(greedy_validities)),
    }
    agg_transformer['free_run_sampled'] = {
        'mean_validity': float(np.mean(sampled_validities)),
        'std_validity': float(np.std(sampled_validities)),
    }
    experiment_result = {
        'label': label, 'bracket_types': bracket_types,
        'n_train': len(train_seqs), 'n_test_per_split': n_test, 'pilot': pilot,
        'distributions': distributions, 'oracle_stats': oracle_stats,
        'baseline_results': baseline_results,
        'transformer_aggregate': agg_transformer,
        'transformer_per_seed': {
            str(rs): {k: v for k, v in transformer_results[rs].items() if k != 'history'}
            for rs in random_seeds
        },
    }
    # Save qualitative samples
    samples_file = os.path.join(OUTPUT_DIR, f'qualitative_samples_{label}.txt')
    with open(samples_file, 'w') as f:
        f.write(f'Qualitative Samples: {label}\n')
        f.write(f'{"="*60}\n\n')
        for rs in random_seeds:
            f.write(f'--- RS={rs} (Greedy) ---\n')
            for s in transformer_results[rs]['free_run_greedy']['samples']:
                status = 'VALID' if s['valid'] else f'INVALID ({s["error_type"]} @ {s["error_pos"]})'
                f.write(f'  {s["string"]}  [{status}]\n')
            f.write(f'\n--- RS={rs} (Sampled T=0.8) ---\n')
            for s in transformer_results[rs]['free_run_sampled']['samples']:
                status = 'VALID' if s['valid'] else f'INVALID ({s["error_type"]} @ {s["error_pos"]})'
                f.write(f'  {s["string"]}  [{status}]\n')
            f.write('\n')
    _print(f'  Saved qualitative samples to {samples_file}')
    return experiment_result


def main():
    parser = argparse.ArgumentParser(description='Dyck Language Experiment')
    parser.add_argument('--pilot', action='store_true')
    parser.add_argument('--dyck1-only', action='store_true')
    parser.add_argument('--dyck2-only', action='store_true')
    parser.add_argument('--random_seeds', nargs='+', type=int, default=RANDOM_SEEDS)
    args = parser.parse_args()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    all_results = {}
    start_time = time.time()
    if not args.dyck2_only:
        all_results['D1'] = run_dyck_experiment(bracket_types=1, random_seeds=args.random_seeds, pilot=args.pilot)
    if not args.dyck1_only:
        all_results['D2'] = run_dyck_experiment(bracket_types=2, random_seeds=args.random_seeds, pilot=args.pilot)
    elapsed = time.time() - start_time
    summary = {'elapsed_seconds': elapsed, 'pilot': args.pilot,
               'random_seeds': args.random_seeds, 'results': all_results}
    with open(os.path.join(OUTPUT_DIR, 'summary.json'), 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    dist_data = {k: r.get('distributions', {}) for k, r in all_results.items()}
    with open(os.path.join(OUTPUT_DIR, 'data_distributions.json'), 'w') as f:
        json.dump(dist_data, f, indent=2, default=str)
    _print(f'\n{"="*80}')
    _print(f'  DYCK EXPERIMENT SUMMARY')
    _print(f'{"="*80}')
    _print(f'  Elapsed: {elapsed:.1f}s')
    _print(f'  Pilot: {args.pilot}')
    _print()
    for key, result in all_results.items():
        _print(f'  --- {key} ---')
        agg = result['transformer_aggregate']
        baselines = result['baseline_results']
        _print(f'  {"Method":<20} {"ID Test":>10} {"Longer OOD":>12} '
               f'{"Deeper OOD":>12} {"Combined OOD":>14}')
        _print(f'  {"-"*70}')
        for bname, bdata in baselines.items():
            row = f'  {bname:<20}'
            for split in ['id_test', 'longer_ood', 'deeper_ood', 'combined_ood']:
                if split in bdata:
                    row += f' {bdata[split]["per_token_accuracy"]:>10.2f}%'
                else:
                    row += f' {"N/A":>10}'
            _print(row)
        row = f'  {"transformer":<20}'
        for split in ['id_test', 'longer_ood', 'deeper_ood', 'combined_ood']:
            if split in agg:
                row += f' {agg[split]["mean_token_acc"]:>10.2f}%'
            else:
                row += f' {"N/A":>10}'
        _print(row)
        row = f'  {"illegal rate":<20}'
        for split in ['id_test', 'longer_ood', 'deeper_ood', 'combined_ood']:
            if split in agg:
                row += f' {agg[split]["mean_illegal_rate"]:>10.2f}%'
            else:
                row += f' {"N/A":>10}'
        _print(row)
        if 'free_run_greedy' in agg:
            _print(f'  Free-run greedy validity: {agg["free_run_greedy"]["mean_validity"]:.2f}%')
        if 'free_run_sampled' in agg:
            _print(f'  Free-run sampled validity: {agg["free_run_sampled"]["mean_validity"]:.2f}%')
        _print()
    _print(f'  Results saved to {OUTPUT_DIR}/')


if __name__ == '__main__':
    main()

## Run Full Experiment

D1 + D2, 3 random seeds each, 10k train / 2k test per split.

Estimated: ~15-20 min on T4 GPU.

In [ ]:
# Run with output going to Google Drive
import os, sys
DRIVE_OUTPUT = '/content/drive/MyDrive/fibonacci-experiment/experiments_v2/exp_dyck'
os.environ['DYCK_OUTPUT_DIR'] = DRIVE_OUTPUT

# Patch OUTPUT_DIR before running
import exp_dyck
exp_dyck.OUTPUT_DIR = DRIVE_OUTPUT

# Override sys.argv so argparse doesn't choke on Jupyter kernel args
sys.argv = ['exp_dyck.py']

exp_dyck.main()

## View Results

In [ ]:
import json

DRIVE_OUTPUT = '/content/drive/MyDrive/fibonacci-experiment/experiments_v2/exp_dyck'

with open(f'{DRIVE_OUTPUT}/summary.json') as f:
    summary = json.load(f)

print(f"Elapsed: {summary['elapsed_seconds']:.1f}s")
print()

for lang_key in ['D1', 'D2']:
    if lang_key not in summary['results']:
        continue
    r = summary['results'][lang_key]
    print(f"=== {lang_key} ===")
    agg = r['transformer_aggregate']
    for split in ['id_test', 'longer_ood', 'deeper_ood', 'combined_ood']:
        a = agg[split]
        print(f"  {split}: token_acc={a['mean_token_acc']:.2f} +/- {a['std_token_acc']:.2f}%, "
              f"illegal={a['mean_illegal_rate']:.2f}%")
    if 'free_run_greedy' in agg:
        print(f"  Free-run greedy validity: {agg['free_run_greedy']['mean_validity']:.2f}%")
    if 'free_run_sampled' in agg:
        print(f"  Free-run sampled validity: {agg['free_run_sampled']['mean_validity']:.2f}%")
    print()

print(f"All results saved to Google Drive: {DRIVE_OUTPUT}")

In [ ]:
# View qualitative samples (from Drive)
DRIVE_OUTPUT = '/content/drive/MyDrive/fibonacci-experiment/experiments_v2/exp_dyck'
for lang in ['D1', 'D2']:
    path = f'{DRIVE_OUTPUT}/qualitative_samples_{lang}.txt'
    try:
        with open(path) as f:
            print(f.read())
    except FileNotFoundError:
        pass

In [ ]:
# List all saved files on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/fibonacci-experiment/experiments_v2/exp_dyck'
import os
for root, dirs, files in os.walk(DRIVE_OUTPUT):
    level = root.replace(DRIVE_OUTPUT, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        print(f'{subindent}{file} ({size:,} bytes)')

In [ ]:
# Post-hoc summary rebuild: add underflow/wrong_type aggregates from manifests
# No retraining — reads existing per-seed manifests and patches summary.json

import json, os
import numpy as np

DRIVE_OUTPUT = '/content/drive/MyDrive/fibonacci-experiment/experiments_v2/exp_dyck'
SEEDS = [42, 123, 7]
SPLITS = ['id_test', 'longer_ood', 'deeper_ood', 'combined_ood']

# 1. Load existing summary
summary_path = os.path.join(DRIVE_OUTPUT, 'summary.json')
with open(summary_path) as f:
    summary = json.load(f)

# 2. Read per-seed manifests and compute aggregates
for lang_key, bracket_types in [('D1', 1), ('D2', 2)]:
    if lang_key not in summary['results']:
        print(f"Skipping {lang_key} (not in summary)")
        continue

    # Load manifests
    manifests = {}
    for rs in SEEDS:
        mpath = os.path.join(DRIVE_OUTPUT, f'{lang_key}_RS{rs}', 'manifest.json')
        with open(mpath) as f:
            manifests[rs] = json.load(f)
        print(f"  Loaded {mpath}")

    agg = summary['results'][lang_key]['transformer_aggregate']

    for split_name in SPLITS:
        tf_key = f'teacher_forced_{split_name}'

        # Extract per-seed rates
        underflow_vals = [manifests[rs]['results'][tf_key]['underflow_rate'] for rs in SEEDS]
        wrong_type_vals = [manifests[rs]['results'][tf_key]['wrong_type_rate'] for rs in SEEDS]

        # Add underflow aggregates (both D1 and D2)
        agg[split_name]['mean_underflow_rate'] = float(np.mean(underflow_vals))
        agg[split_name]['std_underflow_rate'] = float(np.std(underflow_vals))

        # Add wrong_type aggregates (D2 only)
        if bracket_types == 2:
            agg[split_name]['mean_wrong_type_rate'] = float(np.mean(wrong_type_vals))
            agg[split_name]['std_wrong_type_rate'] = float(np.std(wrong_type_vals))

    print(f"\n  {lang_key} aggregates patched.")

# 3. Overwrite summary.json
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nSaved updated summary to {summary_path}")

# 4. Sanity checks
print("\n=== SANITY CHECKS ===")
for lang_key in ['D1', 'D2']:
    if lang_key not in summary['results']:
        continue
    agg = summary['results'][lang_key]['transformer_aggregate']
    print(f"\n--- {lang_key} ---")
    for split_name in SPLITS:
        s = agg[split_name]
        has_underflow = 'mean_underflow_rate' in s
        has_wrong_type = 'mean_wrong_type_rate' in s
        print(f"  {split_name}:")
        if has_underflow:
            print(f"    underflow: {s['mean_underflow_rate']:.4f} +/- {s['std_underflow_rate']:.4f}")
        if has_wrong_type:
            print(f"    wrong_type: {s['mean_wrong_type_rate']:.4f} +/- {s['std_wrong_type_rate']:.4f}")

    # Check: D1 should NOT have wrong_type
    if lang_key == 'D1':
        has_wt = any('mean_wrong_type_rate' in agg[s] for s in SPLITS)
        print(f"  D1 has wrong_type fields: {has_wt} (should be False) {'PASS' if not has_wt else 'FAIL'}")
    # Check: D2 SHOULD have both
    if lang_key == 'D2':
        has_both = all('mean_underflow_rate' in agg[s] and 'mean_wrong_type_rate' in agg[s] for s in SPLITS)
        print(f"  D2 has both underflow+wrong_type: {has_both} (should be True) {'PASS' if has_both else 'FAIL'}")

# Check values differ across splits
for lang_key in ['D1', 'D2']:
    if lang_key not in summary['results']:
        continue
    agg = summary['results'][lang_key]['transformer_aggregate']
    uf_vals = [agg[s]['mean_underflow_rate'] for s in SPLITS if 'mean_underflow_rate' in agg[s]]
    all_same = len(set(round(v, 6) for v in uf_vals)) == 1
    print(f"  {lang_key} underflow values all identical: {all_same} (should be False) {'FAIL' if all_same else 'PASS'}")